# 01.5 — Fill Senate `PartyAB` from party-name lookup

The 2022 Senate CSV (`data/2022_senate_candidates.csv`) has the same column schema as the House CSV, but the `PartyAB` column is blank. Resolve it from `PartyNm` via `data/aec_parties.csv` (canonical names + aliases), then write the filled CSV back so notebook 02 can consume it without doing any party-name mapping itself.

Plain pandas, no Spark.

## 1. Load inputs

In [ ]:
import pandas as pd

SENATE_CSV  = '../data/2022_senate_candidates.csv'
PARTIES_CSV = '../data/aec_parties.csv'

aec_parties = pd.read_csv(PARTIES_CSV, header=0)
senate      = pd.read_csv(SENATE_CSV,  header=0)

print('AEC parties:', len(aec_parties))
print('Senate rows:', len(senate))
senate.head()

## 2. Build the name → PartyAb lookup

Every canonical `party_name` and every entry in the pipe-separated `aliases` column maps to the row's `party_ab`. Lowercased for case-insensitive matching.

In [ ]:
name_to_party_ab = {}
for _, row in aec_parties.iterrows():
    ab = row['party_ab']
    names = [row['party_name']]
    if isinstance(row['aliases'], str) and row['aliases'].strip():
        names.extend(row['aliases'].split('|'))
    for n in names:
        if isinstance(n, str) and n.strip():
            name_to_party_ab[n.strip().lower()] = ab

print('Lookup entries:', len(name_to_party_ab))

## 3. Resolve `PartyAB` for each Senate row

Rules, in order:
1. Blank / NaN `PartyNm` → `IND`.
2. Exact (case-insensitive) match in the lookup → that party's `PartyAb`.
3. Personal-name ticket — `PartyNm` doesn't contain the word "party" — → `IND`.
4. Otherwise → leave blank, surface in the unmapped list for triage.

In [ ]:
def resolve_party_ab(party_nm):
    if not isinstance(party_nm, str) or not party_nm.strip():
        return 'IND'
    key = party_nm.strip().lower()
    if key in name_to_party_ab:
        return name_to_party_ab[key]
    if 'party' not in key:
        return 'IND'
    return None  # genuinely unmapped

senate['PartyAB'] = senate['PartyNm'].apply(resolve_party_ab)

unmapped = senate.loc[senate['PartyAB'].isna(), 'PartyNm'].dropna().unique()
print('Filled rows:', senate['PartyAB'].notna().sum())
print('Unmapped rows:', senate['PartyAB'].isna().sum())
if len(unmapped):
    print('\nUnmapped PartyNm values (triage needed — add to aec_parties.csv aliases):')
    for v in sorted(unmapped):
        print(' -', v)

## 4. Sanity check

In [ ]:
print('PartyAB distribution:')
print(senate['PartyAB'].value_counts(dropna=False))

In [ ]:
# spot-check: a few rows per resolved party
for ab, grp in senate.groupby('PartyAB'):
    print(f'\n=== {ab} ({len(grp)}) ===')
    print(grp[['state', 'PartyAB', 'PartyNm', 'Surname', 'GivenNm']].head(3).to_string(index=False))

## 5. Write back

Overwrite the senate CSV with `PartyAB` filled. Column order is preserved, so notebook 02 can read this file without any further mapping.

In [ ]:
senate.to_csv(SENATE_CSV, index=False)
print('Wrote:', SENATE_CSV)